In [ ]:
import networkx as nx
import numpy as np
import polars as pl

import matplotlib.pyplot as plt
import iplotx as ipx

from climate_attitudes.settings import Config
from climate_attitudes.dataset import Dataset
from climate_attitudes.utils import (
    calculate_stationary_distribution,
    calculate_transition_probabilities,
)

config = Config(_env_file="../.env")

data = Dataset.load(config)

Filter down to participants who are present in waves 1 and 2

In [ ]:
pids = (
    data.participant.filter(wave_1=True, wave_2=True)
    .select("participant_id")
    .collect()
    .to_series()
    .implode()
)

resp = (
    data.response.filter(pl.col("wave") <= 2, pl.col("participant_id").is_in(pids))
    .select(
        "participant_id",
        "wave",
        # pl.col("wave").replace_strict({1: "wave_1", 2: "wave_2"}),
        pl.col("cc1").replace({1: 2, 99: 1}),
        "cvcc4_should",
        pl.col("cc5_world").replace(99, 0),
        "cc6",
    )
    .with_columns(pl.len().over("participant_id").alias("n_waves"))
    .filter(n_waves=2)
    .collect()
)

columns = {
    "Climate change happening": {
        "colname": "cc1",
        "responses": ["No", "Don't know", "Yes"],
    },
    "Climate change anthropogenic ('people should act')": {
        "colname": "cvcc4_should",
        "responses": [
            "Strongly disagree",
            "Disagree",
            "Ambivalent",
            "Agree",
            "Strongly agree",
        ],
    },
    "Climate change worry": {
        "colname": "cc6",
        "responses": [
            "Not at all worried",
            "Not very worried",
            "Somewhat worried",
            "Very worried",
        ],
    },
    "Future generation harm": {
        "colname": "cc5_world",
        "responses": [
            "Don't know",
            "Not at all",
            "Only a little",
            "A moderate amount",
            "A great deal",
        ],
    },
}

In [ ]:
resp_w1 = (
    resp.filter(wave=1)
    .sort(by="participant_id")
    .drop("participant_id", "wave", "n_waves")
    .to_numpy()
)
resp_w2 = (
    resp.filter(wave=2)
    .sort(by="participant_id")
    .drop("participant_id", "wave", "n_waves")
    .to_numpy()
)

resp_w1 = resp_w1 - resp_w1.mean(axis=0)
resp_w2 = resp_w2 - resp_w2.mean(axis=0)

In [ ]:
resp_w2.shape

In [ ]:
# X = np.hstack([resp_w1, np.ones(len(resp_w1))[:, None]])
X = resp_w1
B, *_ = np.linalg.lstsq(X, resp_w2)

# Calculate variance-covariance matrix of residuals
w2_pred = (B @ resp_w1.T).T
resid = resp_w2 - w2_pred
theta = np.corrcoef(resid.T)

# Calculate GGM
theta_inv = np.linalg.inv(theta)
denom = np.sqrt(np.outer(np.diag(theta_inv), np.diag(theta_inv)))
K = -(theta_inv / denom)
K[np.diag_indices_from(K)] = 1.0

In [ ]:
K

In [ ]:
import seaborn as sns

cmap = sns.diverging_palette(230, 20, as_cmap=True)
sns.heatmap(K, center=0, cmap=cmap)

In [ ]:
K[np.diag_indices_from(K)] = 0.0
G = nx.from_numpy_array(K)
layout = nx.spring_layout(G, k=1)

edge_linewidth = {(u, v): 2.0 * z["weight"] for u, v, z in G.edges(data=True)}
edge_labels = [f"{z['weight']:.2f}" for u, v, z in G.edges(data=True)]

network_artist = ipx.network(
    G,
    layout=layout,
    tension=1,
    edge_labels=edge_labels,
    # node_labels=responses,
    edge_linewidth=edge_linewidth,
    edge_curved=True,
    aspect="equal",
    edge_label_bbox=dict(
        edgecolor="black",
        facecolor="white",
        boxstyle="round,pad=0.3",
    ),
    edge_label_rotate=False,
    # ax=ax,
)[0]

In [ ]:
fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(18, 16), constrained_layout=True)

pos = {
    "Climate change happening": np.array([[0, 0], [0.5, 0.87], [1, 0]]),
    # "Climate change anthropogenic ('people should act')": np.array(
    #     [[1, 1], [0, 1], [0.5, 0.5], [0, 0], [1, 0]]
    # ),
    "Climate change anthropogenic ('people should act')": np.array(
        [[1, 0], [1, 1 / 3], [0, 1 / 2], [1, 2 / 3], [1, 1]]
    ),
    "Climate change worry": np.array([[1, 0], [0, 0], [0, 1], [1, 1]]),
    # "Future generation harm": np.array([[0, 0], [1, 0], [1, 0.5], [0, 0.5], [0, 1]]),
    "Future generation harm": np.array(
        [[0, 0], [1, 0], [1, 1 / 3], [1, 2 / 3], [1, 1]]
    ),
}

for i, (dimension, dimension_metadata) in enumerate(columns.items()):
    ax = axes[i // 2, i % 2]
    colname = dimension_metadata["colname"]
    responses = dimension_metadata["responses"]

    transition_probabilities = calculate_transition_probabilities(
        resp, column=colname, start=1, end=2
    )

    if dimension_metadata["colname"].startswith("cc5_"):
        transition_probabilities = np.hstack(
            (np.zeros(5)[:, None], transition_probabilities)
        )

    # Calculate stationary distribution
    mu = calculate_stationary_distribution(transition_probabilities)

    transition_probabilities[transition_probabilities < 0.05] = 0

    G = nx.from_numpy_array(transition_probabilities, create_using=nx.DiGraph)
    edge_linewidth = {(u, v): 2.0 * z["weight"] for u, v, z in G.edges(data=True)}
    edge_labels = [
        f"{z['weight']:.2f}" if z["weight"] > 0.1 else None
        for u, v, z in G.edges(data=True)
    ]

    if dimension in pos:
        layout = pos[dimension]
    else:
        layout = nx.spring_layout(G, k=1)

    with ipx.style.context(
        [
            "hollow",
            {
                "vertex": {
                    "facecolor": mu,
                    "cmap": "Blues",
                    "alpha": 0.5,
                },
                "edge": {
                    "looptension": 2,
                    "loopmaxangle": 10,
                },
            },
        ]
    ):
        network_artist = ipx.network(
            G,
            layout=layout,
            tension=1,
            edge_labels=edge_labels,
            node_labels=responses,
            edge_linewidth=edge_linewidth,
            edge_curved=True,
            aspect="equal",
            edge_label_bbox=dict(
                edgecolor="black",
                facecolor="white",
                boxstyle="round,pad=0.3",
            ),
            edge_label_rotate=False,
            ax=ax,
        )[0]

    plt.colorbar(network_artist.get_vertices(), ax=ax, shrink=0.6)

    ax.set_title(dimension)